In [31]:
import json

with open("/Users/andrewweiland/UCCS_REU/quantum-optimizer-orchestration/brute_force_chains/cheap_optimizers/full_run/one_opt/results_20260610_124820.json") as f:
    data = json.load(f)

print(type(data))

<class 'dict'>


In [32]:
import pandas as pd
df = pd.DataFrame(data["results"])

print(df.iloc[10]["metadata"])
df.iloc[0]

{'chain_name': 'cheap_1_opt_efficient_su2_16_qiskit_standard', 'num_steps': 1, 'steps': [{'name': 'qiskit_standard', 'runner_type': 'qiskit_standard'}], 'step_durations': [0.015418541966937482], 'step_improvements': [0.0], 'total_duration_seconds': 0.017068709013983607}


circuit                                        efficient_su2_10_r2
runner                                                   qiskit_ai
optimizer                                                    chain
label                                              chain_qiskit_ai
metrics          {'depth': 73, 'two_qubit_gates': 18, 'two_qubi...
metadata         {'chain_name': 'cheap_1_opt_efficient_su2_10_r...
artifact_path    brute_force_chains/cheap_optimizers/full_run/o...
Name: 0, dtype: object

In [33]:
df["opt_level"] = df["metadata"].apply(lambda x: x.get("num_steps"))
df["opt_1"] = df["metadata"].apply(
    lambda x: x["steps"][0]["name"] if len(x["steps"]) > 0 else pd.NA
)
df["opt_2"] = df["metadata"].apply(
    lambda x: x["steps"][1]["name"] if len(x["steps"]) > 1 else pd.NA
)
df["opt_3"] = df["metadata"].apply(
    lambda x: x["steps"][2]["name"] if len(x["steps"]) > 2 else pd.NA
)
df["opt_chain"] = df.apply(
    lambda row: "__".join(
        [str(x) for x in [
            row["circuit"],
            row["opt_1"],
            row["opt_2"],
            row["opt_3"]
        ] if pd.notna(x)]
    ),
    axis=1
)
df["runtime"] = df["metadata"].apply(lambda x: x.get("total_duration_seconds"))
df["final_depth"] = df["metrics"].apply(lambda x: x.get("depth"))
df["final_two_qubit_gate_count"] = df["metrics"].apply(lambda x: x.get("two_qubit_gates"))
df["final_two_qubit_depth"] = df["metrics"].apply(lambda x: x.get("two_qubit_depth"))
df["final_total_gates"] = df["metrics"].apply(lambda x: x.get("total_gates"))
df.drop(columns=["metadata", "metrics", "runner", "label", ], inplace=True)
df.head()

,circuit,optimizer,artifact_path,opt_level,opt_1,opt_2,opt_3,opt_chain,runtime,final_depth,final_two_qubit_gate_count,final_two_qubit_depth,final_total_gates
0,efficient_su2_10_r2,chain,brute_force_chains/cheap_optimizers/full_run/o...,1,qiskit_ai,<NA>,<NA>,efficient_su2_10_r2__qiskit_ai,0.764279,73,18,11,198
1,efficient_su2_10_r2,chain,brute_force_chains/cheap_optimizers/full_run/o...,1,qiskit_standard,<NA>,<NA>,efficient_su2_10_r2__qiskit_standard,0.008640,11,18,11,18
2,efficient_su2_10_r2,chain,brute_force_chains/cheap_optimizers/full_run/o...,1,tket,<NA>,<NA>,efficient_su2_10_r2__tket,0.590575,11,18,11,18
3,efficient_su2_12,chain,brute_force_chains/cheap_optimizers/full_run/o...,1,qiskit_ai,<NA>,<NA>,efficient_su2_12__qiskit_ai,0.203203,51,66,30,210
4,efficient_su2_12,chain,brute_force_chains/cheap_optimizers/full_run/o...,1,qiskit_standard,<NA>,<NA>,efficient_su2_12__qiskit_standard,0.221234,29,66,21,162


In [34]:
circuits = pd.read_csv("/Users/andrewweiland/UCCS_REU/quantum-optimizer-orchestration/dataset_analysis/circuits.csv")
circuits.head()

,Unnamed: 0,id,name,category,source,qasm_path,num_qubits,initial_depth,initial_two_qubit_gates,initial_two_qubit_depth,initial_total_gates,gate_density,two_qubit_ratio,created_at,optimizer_chain,original_circuit
0,0,1,efficient_su2_10_r2,efficient_su2,local,benchmarks/ai_transpile/qasm/efficient_su2_10_...,10,17,18,11,78,7.800000,0.230769,2026-03-05 20:39:31,NaN,efficient_su2_10_r2
1,1,2,efficient_su2_12,efficient_su2,local,benchmarks/ai_transpile/qasm/efficient_su2_12....,12,25,66,21,114,9.500000,0.578947,2026-03-05 20:39:31,NaN,efficient_su2_12
2,2,3,efficient_su2_12_r2,efficient_su2,local,benchmarks/ai_transpile/qasm/efficient_su2_12_...,12,19,22,13,94,7.833333,0.234043,2026-03-05 20:39:31,NaN,efficient_su2_12_r2
3,3,4,efficient_su2_16,efficient_su2,local,benchmarks/ai_transpile/qasm/efficient_su2_16....,16,33,120,29,184,11.500000,0.652174,2026-03-05 20:39:31,NaN,efficient_su2_16
4,4,5,efficient_su2_8,efficient_su2,local,benchmarks/ai_transpile/qasm/efficient_su2_8.qasm,8,27,56,21,104,13.000000,0.538462,2026-03-05 20:39:31,NaN,efficient_su2_8


In [35]:
df["parent_circuit_id"] = df["circuit"].map(
    circuits.set_index("name")["id"]
)
df.head()
df["parent_circuit"] = df["circuit"]
df["original_circuit_id"] = df["circuit"].map(
    circuits.set_index("name")["id"]
)
df["original_circuit"] = df["circuit"]
df["original_category"] = df["circuit"].map(
    circuits.set_index("name")["category"]
)
df["original_circuit_path"] = df["circuit"].map(
    circuits.set_index("name")["qasm_path"]
)

df["chain_name"] = df.apply(
    lambda row: "__".join(
        [str(x) for x in [
            row["opt_1"],
            row["opt_2"],
            row["opt_3"]
        ] if pd.notna(x)]
    ),
    axis=1
)

df["initial_num_qubits"] = df["circuit"].map(
    circuits.set_index("name")["num_qubits"]
)
df["initial_depth"] = df["circuit"].map(
    circuits.set_index("name")["initial_depth"]
)
df["initial_two_qubit_gates"] = df["circuit"].map(
    circuits.set_index("name")["initial_two_qubit_gates"]
)
df["initial_total_gates"] = df["circuit"].map(
    circuits.set_index("name")["initial_total_gates"]
)
df["final_num_qubits"] = df["initial_num_qubits"]

df = df[
    [
        "parent_circuit_id",
        "parent_circuit",
        "original_circuit_id",
        "original_circuit",
        "original_category",
        "original_circuit_path",
        "chain_name",
        "opt_level",
        "opt_1",
        "opt_2",
        "opt_3",
        "artifact_path",
        "initial_num_qubits",
        "initial_depth",
        "initial_two_qubit_gates",
        "initial_total_gates",
        "final_num_qubits",
        "final_depth",
        "final_two_qubit_gate_count",
        "final_total_gates",
        "runtime",
        "opt_chain"
    ]
]

df.head()

,parent_circuit_id,parent_circuit,original_circuit_id,original_circuit,original_category,original_circuit_path,chain_name,opt_level,opt_1,opt_2,opt_3,artifact_path,initial_num_qubits,initial_depth,initial_two_qubit_gates,initial_total_gates,final_num_qubits,final_depth,final_two_qubit_gate_count,final_total_gates,runtime,opt_chain
0,1,efficient_su2_10_r2,1,efficient_su2_10_r2,efficient_su2,benchmarks/ai_transpile/qasm/efficient_su2_10_...,qiskit_ai,1,qiskit_ai,<NA>,<NA>,brute_force_chains/cheap_optimizers/full_run/o...,10,17,18,78,10,73,18,198,0.764279,efficient_su2_10_r2__qiskit_ai
1,1,efficient_su2_10_r2,1,efficient_su2_10_r2,efficient_su2,benchmarks/ai_transpile/qasm/efficient_su2_10_...,qiskit_standard,1,qiskit_standard,<NA>,<NA>,brute_force_chains/cheap_optimizers/full_run/o...,10,17,18,78,10,11,18,18,0.008640,efficient_su2_10_r2__qiskit_standard
2,1,efficient_su2_10_r2,1,efficient_su2_10_r2,efficient_su2,benchmarks/ai_transpile/qasm/efficient_su2_10_...,tket,1,tket,<NA>,<NA>,brute_force_chains/cheap_optimizers/full_run/o...,10,17,18,78,10,11,18,18,0.590575,efficient_su2_10_r2__tket
3,2,efficient_su2_12,2,efficient_su2_12,efficient_su2,benchmarks/ai_transpile/qasm/efficient_su2_12....,qiskit_ai,1,qiskit_ai,<NA>,<NA>,brute_force_chains/cheap_optimizers/full_run/o...,12,25,66,114,12,51,66,210,0.203203,efficient_su2_12__qiskit_ai
4,2,efficient_su2_12,2,efficient_su2_12,efficient_su2,benchmarks/ai_transpile/qasm/efficient_su2_12....,qiskit_standard,1,qiskit_standard,<NA>,<NA>,brute_force_chains/cheap_optimizers/full_run/o...,12,25,66,114,12,29,66,162,0.221234,efficient_su2_12__qiskit_standard


In [42]:
df.to_csv("cheap_single_opts.csv")

In [36]:
pd.set_option("display.max_columns", None)
all_possible_chains = pd.read_csv("/Users/andrewweiland/UCCS_REU/quantum-optimizer-orchestration/dataset_analysis/all_possible_chains.csv")
all_possible_chains.head()

,id,opt_chain,parent_circuit_id,parent_circuit,parent_path,original_circuit_id,original_circuit,original_category,original_circuit_path,chain_name,opt_level,opt_1,opt_2,opt_3,artifact_path,initial_num_qubits,initial_depth,initial_two_qubit_gates,initial_total_gates,final_num_qubits,final_depth,final_two_qubit_gates,final_total_gates,total_runtime_seconds
0,0,efficient_su2_10_r2__qiskit_ai,1,efficient_su2_10_r2,NaN,1,efficient_su2_10_r2,efficient_su2,benchmarks/ai_transpile/qasm/efficient_su2_10_...,NaN,1,qiskit_ai,NaN,NaN,NaN,10,17,18,78,0,0,0,0,0.0
1,1,efficient_su2_10_r2__qiskit_standard,1,efficient_su2_10_r2,NaN,1,efficient_su2_10_r2,efficient_su2,benchmarks/ai_transpile/qasm/efficient_su2_10_...,NaN,1,qiskit_standard,NaN,NaN,NaN,10,17,18,78,0,0,0,0,0.0
2,2,efficient_su2_10_r2__tket,1,efficient_su2_10_r2,NaN,1,efficient_su2_10_r2,efficient_su2,benchmarks/ai_transpile/qasm/efficient_su2_10_...,NaN,1,tket,NaN,NaN,NaN,10,17,18,78,0,0,0,0,0.0
3,3,efficient_su2_10_r2__wisq_rules,1,efficient_su2_10_r2,NaN,1,efficient_su2_10_r2,efficient_su2,benchmarks/ai_transpile/qasm/efficient_su2_10_...,NaN,1,wisq_rules,NaN,NaN,NaN,10,17,18,78,0,0,0,0,0.0
4,4,efficient_su2_10_r2__wisq_bqskit,1,efficient_su2_10_r2,NaN,1,efficient_su2_10_r2,efficient_su2,benchmarks/ai_transpile/qasm/efficient_su2_10_...,NaN,1,wisq_bqskit,NaN,NaN,NaN,10,17,18,78,0,0,0,0,0.0


In [39]:
df

,parent_circuit_id,parent_circuit,original_circuit_id,original_circuit,original_category,original_circuit_path,chain_name,opt_level,opt_1,opt_2,opt_3,artifact_path,initial_num_qubits,initial_depth,initial_two_qubit_gates,initial_total_gates,final_num_qubits,final_depth,final_two_qubit_gate_count,final_total_gates,runtime,opt_chain
0,1,efficient_su2_10_r2,1,efficient_su2_10_r2,efficient_su2,benchmarks/ai_transpile/qasm/efficient_su2_10_...,qiskit_ai,1,qiskit_ai,<NA>,<NA>,brute_force_chains/cheap_optimizers/full_run/o...,10,17,18,78,10,73,18,198,0.764279,efficient_su2_10_r2__qiskit_ai
1,1,efficient_su2_10_r2,1,efficient_su2_10_r2,efficient_su2,benchmarks/ai_transpile/qasm/efficient_su2_10_...,qiskit_standard,1,qiskit_standard,<NA>,<NA>,brute_force_chains/cheap_optimizers/full_run/o...,10,17,18,78,10,11,18,18,0.008640,efficient_su2_10_r2__qiskit_standard
2,1,efficient_su2_10_r2,1,efficient_su2_10_r2,efficient_su2,benchmarks/ai_transpile/qasm/efficient_su2_10_...,tket,1,tket,<NA>,<NA>,brute_force_chains/cheap_optimizers/full_run/o...,10,17,18,78,10,11,18,18,0.590575,efficient_su2_10_r2__tket
3,2,efficient_su2_12,2,efficient_su2_12,efficient_su2,benchmarks/ai_transpile/qasm/efficient_su2_12....,qiskit_ai,1,qiskit_ai,<NA>,<NA>,brute_force_chains/cheap_optimizers/full_run/o...,12,25,66,114,12,51,66,210,0.203203,efficient_su2_12__qiskit_ai
4,2,efficient_su2_12,2,efficient_su2_12,efficient_su2,benchmarks/ai_transpile/qasm/efficient_su2_12....,qiskit_standard,1,qiskit_standard,<NA>,<NA>,brute_force_chains/cheap_optimizers/full_run/o...,12,25,66,114,12,29,66,162,0.221234,efficient_su2_12__qiskit_standard
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
318,110,square_heisenberg_N4,110,square_heisenberg_N4,square-heisenberg,benchmarks/ai_transpile/qasm/square-heisenberg...,qiskit_standard,1,qiskit_standard,<NA>,<NA>,brute_force_chains/cheap_optimizers/full_run/o...,4,121,48,172,4,35,12,57,0.018435,square_heisenberg_N4__qiskit_standard
319,110,square_heisenberg_N4,110,square_heisenberg_N4,square-heisenberg,benchmarks/ai_transpile/qasm/square-heisenberg...,tket,1,tket,<NA>,<NA>,brute_force_chains/cheap_optimizers/full_run/o...,4,121,48,172,4,40,12,66,1.571808,square_heisenberg_N4__tket
320,111,square_heisenberg_N9,111,square_heisenberg_N9,square-heisenberg,benchmarks/ai_transpile/qasm/square-heisenberg...,qiskit_ai,1,qiskit_ai,<NA>,<NA>,brute_force_chains/cheap_optimizers/full_run/o...,9,241,144,513,9,641,144,1665,5.767150,square_heisenberg_N9__qiskit_ai
321,111,square_heisenberg_N9,111,square_heisenberg_N9,square-heisenberg,benchmarks/ai_transpile/qasm/square-heisenberg...,qiskit_standard,1,qiskit_standard,<NA>,<NA>,brute_force_chains/cheap_optimizers/full_run/o...,9,241,144,513,9,69,36,168,0.039671,square_heisenberg_N9__qiskit_standard


In [40]:
two_opts = all_possible_chains[all_possible_chains["opt_level"] == 2].copy()
two_opts["parent_path"] = two_opts["parent_circuit"].map(
    df.set_index("opt_chain")["artifact_path"]
)
two_opts

,id,opt_chain,parent_circuit_id,parent_circuit,parent_path,original_circuit_id,original_circuit,original_category,original_circuit_path,chain_name,opt_level,opt_1,opt_2,opt_3,artifact_path,initial_num_qubits,initial_depth,initial_two_qubit_gates,initial_total_gates,final_num_qubits,final_depth,final_two_qubit_gates,final_total_gates,total_runtime_seconds
5,5,efficient_su2_10_r2__qiskit_ai__qiskit_ai,0,efficient_su2_10_r2__qiskit_ai,brute_force_chains/cheap_optimizers/full_run/o...,1,efficient_su2_10_r2,efficient_su2,benchmarks/ai_transpile/qasm/efficient_su2_10_...,NaN,2,qiskit_ai,qiskit_ai,NaN,NaN,10,17,18,78,0,0,0,0,0.0
6,6,efficient_su2_10_r2__qiskit_ai__qiskit_standard,0,efficient_su2_10_r2__qiskit_ai,brute_force_chains/cheap_optimizers/full_run/o...,1,efficient_su2_10_r2,efficient_su2,benchmarks/ai_transpile/qasm/efficient_su2_10_...,NaN,2,qiskit_ai,qiskit_standard,NaN,NaN,10,17,18,78,0,0,0,0,0.0
7,7,efficient_su2_10_r2__qiskit_ai__tket,0,efficient_su2_10_r2__qiskit_ai,brute_force_chains/cheap_optimizers/full_run/o...,1,efficient_su2_10_r2,efficient_su2,benchmarks/ai_transpile/qasm/efficient_su2_10_...,NaN,2,qiskit_ai,tket,NaN,NaN,10,17,18,78,0,0,0,0,0.0
8,8,efficient_su2_10_r2__qiskit_ai__wisq_rules,0,efficient_su2_10_r2__qiskit_ai,brute_force_chains/cheap_optimizers/full_run/o...,1,efficient_su2_10_r2,efficient_su2,benchmarks/ai_transpile/qasm/efficient_su2_10_...,NaN,2,qiskit_ai,wisq_rules,NaN,NaN,10,17,18,78,0,0,0,0,0.0
9,9,efficient_su2_10_r2__qiskit_ai__wisq_bqskit,0,efficient_su2_10_r2__qiskit_ai,brute_force_chains/cheap_optimizers/full_run/o...,1,efficient_su2_10_r2,efficient_su2,benchmarks/ai_transpile/qasm/efficient_su2_10_...,NaN,2,qiskit_ai,wisq_bqskit,NaN,NaN,10,17,18,78,0,0,0,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17075,17075,square_heisenberg_N9__wisq_bqskit__qiskit_ai,17054,square_heisenberg_N9__wisq_bqskit,NaN,111,square_heisenberg_N9,square-heisenberg,benchmarks/ai_transpile/qasm/square-heisenberg...,NaN,2,wisq_bqskit,qiskit_ai,NaN,NaN,9,241,144,513,0,0,0,0,0.0
17076,17076,square_heisenberg_N9__wisq_bqskit__qiskit_stan...,17054,square_heisenberg_N9__wisq_bqskit,NaN,111,square_heisenberg_N9,square-heisenberg,benchmarks/ai_transpile/qasm/square-heisenberg...,NaN,2,wisq_bqskit,qiskit_standard,NaN,NaN,9,241,144,513,0,0,0,0,0.0
17077,17077,square_heisenberg_N9__wisq_bqskit__tket,17054,square_heisenberg_N9__wisq_bqskit,NaN,111,square_heisenberg_N9,square-heisenberg,benchmarks/ai_transpile/qasm/square-heisenberg...,NaN,2,wisq_bqskit,tket,NaN,NaN,9,241,144,513,0,0,0,0,0.0
17078,17078,square_heisenberg_N9__wisq_bqskit__wisq_rules,17054,square_heisenberg_N9__wisq_bqskit,NaN,111,square_heisenberg_N9,square-heisenberg,benchmarks/ai_transpile/qasm/square-heisenberg...,NaN,2,wisq_bqskit,wisq_rules,NaN,NaN,9,241,144,513,0,0,0,0,0.0


In [41]:
two_opts[two_opts["parent_path"].notna()]

,id,opt_chain,parent_circuit_id,parent_circuit,parent_path,original_circuit_id,original_circuit,original_category,original_circuit_path,chain_name,opt_level,opt_1,opt_2,opt_3,artifact_path,initial_num_qubits,initial_depth,initial_two_qubit_gates,initial_total_gates,final_num_qubits,final_depth,final_two_qubit_gates,final_total_gates,total_runtime_seconds
5,5,efficient_su2_10_r2__qiskit_ai__qiskit_ai,0,efficient_su2_10_r2__qiskit_ai,brute_force_chains/cheap_optimizers/full_run/o...,1,efficient_su2_10_r2,efficient_su2,benchmarks/ai_transpile/qasm/efficient_su2_10_...,NaN,2,qiskit_ai,qiskit_ai,NaN,NaN,10,17,18,78,0,0,0,0,0.0
6,6,efficient_su2_10_r2__qiskit_ai__qiskit_standard,0,efficient_su2_10_r2__qiskit_ai,brute_force_chains/cheap_optimizers/full_run/o...,1,efficient_su2_10_r2,efficient_su2,benchmarks/ai_transpile/qasm/efficient_su2_10_...,NaN,2,qiskit_ai,qiskit_standard,NaN,NaN,10,17,18,78,0,0,0,0,0.0
7,7,efficient_su2_10_r2__qiskit_ai__tket,0,efficient_su2_10_r2__qiskit_ai,brute_force_chains/cheap_optimizers/full_run/o...,1,efficient_su2_10_r2,efficient_su2,benchmarks/ai_transpile/qasm/efficient_su2_10_...,NaN,2,qiskit_ai,tket,NaN,NaN,10,17,18,78,0,0,0,0,0.0
8,8,efficient_su2_10_r2__qiskit_ai__wisq_rules,0,efficient_su2_10_r2__qiskit_ai,brute_force_chains/cheap_optimizers/full_run/o...,1,efficient_su2_10_r2,efficient_su2,benchmarks/ai_transpile/qasm/efficient_su2_10_...,NaN,2,qiskit_ai,wisq_rules,NaN,NaN,10,17,18,78,0,0,0,0,0.0
9,9,efficient_su2_10_r2__qiskit_ai__wisq_bqskit,0,efficient_su2_10_r2__qiskit_ai,brute_force_chains/cheap_optimizers/full_run/o...,1,efficient_su2_10_r2,efficient_su2,benchmarks/ai_transpile/qasm/efficient_su2_10_...,NaN,2,qiskit_ai,wisq_bqskit,NaN,NaN,10,17,18,78,0,0,0,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17065,17065,square_heisenberg_N9__tket__qiskit_ai,17052,square_heisenberg_N9__tket,brute_force_chains/cheap_optimizers/full_run/o...,111,square_heisenberg_N9,square-heisenberg,benchmarks/ai_transpile/qasm/square-heisenberg...,NaN,2,tket,qiskit_ai,NaN,NaN,9,241,144,513,0,0,0,0,0.0
17066,17066,square_heisenberg_N9__tket__qiskit_standard,17052,square_heisenberg_N9__tket,brute_force_chains/cheap_optimizers/full_run/o...,111,square_heisenberg_N9,square-heisenberg,benchmarks/ai_transpile/qasm/square-heisenberg...,NaN,2,tket,qiskit_standard,NaN,NaN,9,241,144,513,0,0,0,0,0.0
17067,17067,square_heisenberg_N9__tket__tket,17052,square_heisenberg_N9__tket,brute_force_chains/cheap_optimizers/full_run/o...,111,square_heisenberg_N9,square-heisenberg,benchmarks/ai_transpile/qasm/square-heisenberg...,NaN,2,tket,tket,NaN,NaN,9,241,144,513,0,0,0,0,0.0
17068,17068,square_heisenberg_N9__tket__wisq_rules,17052,square_heisenberg_N9__tket,brute_force_chains/cheap_optimizers/full_run/o...,111,square_heisenberg_N9,square-heisenberg,benchmarks/ai_transpile/qasm/square-heisenberg...,NaN,2,tket,wisq_rules,NaN,NaN,9,241,144,513,0,0,0,0,0.0
